# lunar-scripts-jp —— JP 私服客户端制作（Colab）

仓库: https://github.com/lovelyzy7/lunar-scripts-jp

流程: 安装依赖 → 准备原始 JP APK → 解包 → 打补丁 → 重建/签名 → 下载 APK

In [ ]:
REPO = 'https://github.com/lovelyzy7/lunar-scripts-jp.git'
!git clone -q $REPO
%cd lunar-scripts-jp
!pip -q install -r requirements.txt
!python android/build_apk.py tools
!java -version

In [ ]:
# 原始 JP APK：可放 Google Drive，也可现场上传
from google.colab import drive, files
drive.mount('/content/drive')
APK = '/content/drive/MyDrive/nierspjp-3.7.1.apk'   # ← 改成你的路径
import os; assert os.path.exists(APK), APK
print('APK:', APK, os.path.getsize(APK)//1048576, 'MB')

In [ ]:
!python android/build_apk.py unpack $APK work/jp-final

In [ ]:
# 打补丁：端口按你的服务端配置
HOST = '203.0.113.10'      # <-- 你的服务器地址（公网 IP / 域名）
GRPC = HOST + ':8003'      # gRPC 游戏服务
HTTP = HOST + ':8080'      # HTTP / CDN
AUTH = HOST + ':3000'      # 登录 / OAuth 回调
!python android/patch_apk_jp.py work/jp-final --grpc-addr $GRPC --http-addr $HTTP --auth-host $AUTH


In [ ]:
!python android/build_apk.py build work/jp-final /content/drive/MyDrive/nierspjp-jp-modded.apk
# 直接下载（大文件建议用上一行的 Drive 输出）
# files.download('/content/drive/MyDrive/nierspjp-jp-modded.apk')

### 备注
- Colab 会话是临时的：输入/输出建议都放 Google Drive。
- 不需要 frida（设备侧诊断工具，非必需）。
- APK 工具（apktool + uber-apk-signer）由 `android/build_apk.py tools` 自动下载到 `work/tools/`。